# Preprocessing StereoQueerEval Training

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

input_path = Path("Analisis/StereoQueerEval_Training.xlsx")
if not input_path.exists():
    input_path = Path("StereoQueerEval_Training.xlsx")

output_dir = Path("data") if input_path.parent.name != "Analisis" else input_path.parent.parent / "data"
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(input_path, sheet_name="ALL_DATA")
df = df.drop(columns=["stereotype", "target"])

folds = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
fold_id = pd.Series(index=df.index, dtype="int")
groups = df["yt_title"].fillna("").astype(str).str.strip()
stratification_label = df["language"].astype(str) + "__" + df["hate_speech"].astype(str)

for i, (_, test_index) in enumerate(folds.split(df, stratification_label, groups)):
    fold_id.iloc[test_index] = i

splits = {
    "train": df.loc[~fold_id.isin([0, 1, 2])],
    "val": df.loc[fold_id == 0],
    "test": df.loc[fold_id.isin([1, 2])],
}

split_summary = pd.DataFrame({
    "jumlah": {name: len(data) for name, data in splits.items()},
    "proporsi_total_%": {
        name: round(len(data) / len(df) * 100, 2) for name, data in splits.items()
    },
})
print("Proporsi split (%):")
print(split_summary.to_string())

class_counts = pd.DataFrame({
    name: data["hate_speech"].value_counts() for name, data in splits.items()
}).T.fillna(0).astype(int)
class_proportions = class_counts.div(class_counts.sum(axis=1), axis=0).mul(100).round(2)
class_proportions.loc["full_data"] = df["hate_speech"].value_counts(normalize=True).mul(100).round(2)

print("\nJumlah tiap kelas hate_speech:")
print(class_counts.to_string())
print("\nProporsi kelas dalam tiap subset (%):")
print(class_proportions.to_string())

language_counts = pd.DataFrame({
    name: data["language"].value_counts() for name, data in splits.items()
}).T.fillna(0).astype(int)
language_proportions = language_counts.div(language_counts.sum(axis=1), axis=0).mul(100).round(2)
language_proportions.loc["full_data"] = df["language"].value_counts(normalize=True).mul(100).round(2)
print("\nProporsi bahasa dalam tiap subset (%):")
print(language_proportions.to_string())

for name, data in splits.items():
    print(f"\nProporsi kombinasi bahasa dan hate_speech ? {name} (%):")
    print((pd.crosstab(data["language"], data["hate_speech"], normalize="all") * 100).round(2).to_string())

for name, data in splits.items():
    data.to_excel(output_dir / f"{name}.xlsx", index=False)


Proporsi split (%):
       jumlah  proporsi_total_%
train    5339             70.00
val       764             10.02
test     1524             19.98

Jumlah tiap kelas hate_speech:
hate_speech    no  yes_explicit  yes_implicit
train        2591          1461          1287
val           366           214           184
test          737           418           369

Proporsi kelas dalam tiap subset (%):
hate_speech     no  yes_explicit  yes_implicit
train        48.53         27.36         24.11
val          47.91         28.01         24.08
test         48.36         27.43         24.21
full_data    48.43         27.44         24.12

Proporsi bahasa dalam tiap subset (%):
language      EN     IT     NL
train      39.18  31.45  29.37
val        39.01  31.28  29.71
test       39.30  31.63  29.07
full_data  39.19  31.47  29.34

Proporsi kombinasi bahasa dan hate_speech ? train (%):
hate_speech     no  yes_explicit  yes_implicit
language                                      
EN           18.4